# Candidate sets change a softmax weight

How much does the **mean softmax weight of four designated original pairs** change
when two public candidate columns are added? Compare independent sigmoid scores.
This is a fixed-logit analytic exercise, not calibrated native probabilities or a
comparison of full trained CLIP/SigLIP models. TRAIN: none; DEV: public distinct
fixtures; TEST: none. Public source and formulas are exposed. Record prior exposure.
Target construction/execution is currently **disabled pending BV03**.

By the end, implement both readouts, distinguish competition from independent
pair scoring, commit an exposure-aware numerical forecast, and interpret controls.
Cold open: imagine matching a red-bicycle query against three captions. Add two
extra captions while keeping every old pair logit fixed. Should the old bicycle
pair lose softmax weight? Must its independent score change? Explain your causal
prediction first; the public four-row exercise asks how large the mean change is.


In [ ]:
from mastery_hub.units.clip_prediction import lesson
import torch as t


## Public protocol (plaintext, no target tensors)

```json
{
  "id": "12.1-candidate-set-normalization-v1",
  "row_ids": [
    "q0",
    "q1",
    "q2",
    "q3"
  ],
  "candidate_ids": [
    "c0",
    "c1",
    "c2",
    "c3",
    "c4"
  ],
  "base_ids": [
    "c0",
    "c1",
    "c2"
  ],
  "selected_ids": [
    "c1",
    "c2",
    "c0",
    "c1"
  ],
  "logit_rule": "(((7*i + 5*j + 3) mod 17) - 8) / 4; 0<=i<4, 0<=j<5",
  "raw_logits": [
    [
      -1.25,
      0.0,
      1.25,
      -1.75,
      -0.5
    ],
    [
      0.5,
      1.75,
      -1.25,
      0.0,
      1.25
    ],
    [
      -2.0,
      -0.75,
      0.5,
      1.75,
      -1.25
    ],
    [
      -0.25,
      1.0,
      -2.0,
      -0.75,
      0.5
    ]
  ],
  "permutation": [
    4,
    1,
    3,
    0,
    2
  ],
  "shift": 1,
  "dtype": "float64",
  "device": "cpu",
  "threads": 1,
  "conditions": [
    "baseline",
    "expansion",
    "unchanged_baseline",
    "permuted_expansion",
    "shifted_baseline",
    "shifted_expansion"
  ],
  "kinds": [
    "softmax",
    "sigmoid"
  ],
  "max_calls": 12,
  "timeout_seconds": 30,
  "rtol": 1e-10,
  "atol": 1e-12,
  "metric": "mean(selected_expanded_softmax-selected_baseline_softmax)",
  "units": "weight units; 0.01 equals one percentage point",
  "roles": {
    "TRAIN": "none",
    "DEV": "exposed distinct literal fixtures",
    "TEST": "none"
  },
  "target_status": "UNCONSTRUCTED_UNEXECUTED",
  "source_pins": {
    "infrastructure/chapters/chapter12_vlm_interpretability/master_12_1.py": "47c1b0aafec6115a759f4ee7ba52aa407f9e9af4dff28599454cead7e3f95df3",
    "chapter12_vlm_interpretability/exercises/part1_clip_siglip/tests.py": "88fe498d1e8cd43b3e875352712a2fc310541ce05d17c2cdc4baca359eef0e18",
    "mastery_hub/units/extension_predictions/lesson.py": "fe0e58e0577b1c348537ab7f33cee9aca4d6c2ae24b5c8562822fb0424cb8659",
    "mastery_hub/units/ssd_prediction/lesson.py": "431e5cc0ab598e11b98f033038d958ab9d4df603201d6d924303d0a049a1a659",
    "mastery_hub/units/sae_prediction/records.py": "2fece05ac32b881f929b7697fe5adade55ab871798b2adfd9e35a24f0360dbca",
    "scripts/build_sae_prediction_notebook.py": "dbd9a95ef35b6c234e80bee4933250cfbaa2348b01936deed22f4f3ae6f5e437"
  }
}

```

Delta uses weight units: 0.01 is one percentage point. All four rows remain in order. Signed forecast error is observed Delta minus predicted Delta. The input rule is public, but no target score or plot has been computed here.

In [ ]:
# Complete this before opening solutions or measuring the target.
manual_forecast = dict(prediction=None, confidence=None, reasoning='', revision='')
prior_exposure = None  # Choose 'seen', 'unknown', or 'self_reported_not_seen'.
# In reasoning, disclose prior target/reference/output or related-solution exposure.


## Implement the readout (about five minutes)

Implement the original `zero_shot_probabilities(logits, kind)` interface.
`kind='softmax'`: normalize over candidate columns in each row.
`kind='sigmoid'`: transform each pair independently. Reject an unknown kind.
Output keeps input shape, CPU device and float64 dtype. No reference fallback.

<details><summary>Help</summary>
The last axis contains candidates. Sigmoid has no candidate-sum denominator.
</details>

Expected immediate-test outputs: shape `(2, 3)`, softmax row sums one, all-zero
sigmoid row equal to one half, and unknown kind raises `ValueError`.


In [ ]:
METHOD_SOURCE = """def zero_shot_probabilities(logits, kind):
    raise NotImplementedError("Implement both readouts")
"""
# This is your editable method. The explicit DEV cell below defines and calls it.


In [ ]:
# Explicit tiny DEV only: these are not the public four-row target.
lesson.validate_source(METHOD_SOURCE)
namespace = {'t': t, 'torch': t}
exec(compile(METHOD_SOURCE, '<my CLIP DEV method>', 'exec'), namespace)
my_readout = namespace['zero_shot_probabilities']
x_dev = t.tensor([[0., 0., 0.], [1., -2., 0.5]], dtype=t.float64, device='cpu')
y_soft = my_readout(x_dev.clone(), 'softmax')
y_sig = my_readout(x_dev.clone(), 'sigmoid')
t.testing.assert_close(y_soft, x_dev.softmax(-1))
t.testing.assert_close(y_sig, x_dev.sigmoid())
assert y_soft.shape == y_sig.shape == x_dev.shape
t.testing.assert_close(y_soft.sum(-1), t.ones(2, dtype=t.float64))
t.testing.assert_close(y_sig[0], t.full((3,), 0.5, dtype=t.float64))
try:
    my_readout(x_dev.clone(), 'unknown')
except ValueError:
    pass
else:
    raise AssertionError('Unknown kind must raise ValueError')
print('Immediate DEV tests passed; target remains unexecuted.')


## Commit your completed method and numerical forecast

Keep the original manual forecast. If DEV feedback changed your prediction,
confidence or reason, explain the revision explicitly. A software commitment hashes
your actual method, public protocol and recoverable source snapshots. It does not
prove an adversarial chronology or unseen mastery. Changing source requires a new
commitment, but does **not** reset an already-started session attempt.


In [ ]:
prediction = manual_forecast['prediction']
confidence = manual_forecast['confidence']
reasoning = manual_forecast['reasoning']
identity = lesson.freeze_inputs(METHOD_SOURCE)
commitment = lesson.commit_prediction(
    identity, METHOD_SOURCE, prediction, confidence, reasoning, prior_exposure,
    manual_forecast=manual_forecast,
)


In [ ]:
# Optional public solution after commitment; never installs code.
from IPython.display import Markdown, display
display(Markdown(lesson.reveal_methods(commitment)))


## Future run and visible result

Six conditions × two readouts = 12 calls: baseline, expansion, repeated baseline,
coherent candidate permutation, shifted baseline, shifted expansion. Every condition
and control passes before a mean or accepted figure. Identity and permutation must
preserve results; sigmoid must preserve fixed-pair scores under expansion; softmax
must preserve rows under a common shift. Sigmoid's shift response is a diagnostic.
Expected future signature: paired per-query before/after bars, mean changes, forecast
and input heatmap. No numerical target answer is supplied. A wrong prediction is
useful evidence, not a failed engineering test. Whole-cohort failures stay visible.


In [ ]:
receipt = None  # Clear stale success before prerequisites or running.
# BV03 is deferred. After explicit authority and reviewed enabling change only:
# receipt = lesson.run_experiment(commitment, METHOD_SOURCE)
# Failed attempts raise AttemptFailure with its preserved .receipt; never auto-retry.


In [ ]:
if receipt is not None:
    lesson.plot_receipt(receipt)


<details><summary>Interpretation and limits</summary>
Adding alternatives changes a softmax denominator without changing original logits.
An independent sigmoid has no such denominator. This algebra does not show that a
trained SigLIP is calibrated, that CLIP ignores objects, or that either model has
better retrieval accuracy. Read the existing 12.1 loss/readout discussion and its
paper connections: [CLIP](https://arxiv.org/abs/2103.00020) and
[SigLIP](https://arxiv.org/abs/2303.15343). Existing real-image results remain historical, separate evidence.
</details>

## Reflect, then Try It Yourself
Explain the largest mismatch and what the controls rule out. Propose changing the
candidate set or common logit shift; commit a new prediction before changing inputs.
This first companion does not authorize or execute a second fixed target. Keep all
old commitments and failures. Replay and private export never run the learner.


In [ ]:
# Try It Yourself: make this prospective prediction editable before any new run.
play_hypothesis = dict(changed_condition='', prediction=None, confidence=None,
                       reasoning='')
# No changed-input construction or normalization is authorized by this play cell.
# reflection = lesson.reflect(receipt, explanation='', changed_condition='',
#                             prediction=None, confidence=None, reasoning='')
# lesson.export_record(receipt, '/absolute/private/path/new-receipt.json')
# lesson.export_attempt(reflection, '/absolute/private/path/new-reflection.json')
# replayed = lesson.replay(receipt)  # Stored observations only, no method execution.
